# Probe scan — locating misfiled structured content

Induced shapes become lexicalised probes (letter slots filled with tokens observed in the structured fields), every hit is verified by the group's tier-1 parser, and each verified span is compared against the record's own structured values. Output: `probe_candidates.parquet`, one row per (node, group, candidate span).

In [1]:
from pathlib import Path
import re

import polars as pl
from codecarbon import EmissionsTracker
from mds_data_model.introspection import (
    measurement_fields,
    vocab_fields,
    date_fields,
    monetary_fields,
    free_text_fields
)

from mds_norm.parsers.parse_dates import parse_date
from mds_norm.parsers.parse_dimensions import parse_dimensions
from mds_norm.parsers.parse_monetary import parse_monetary

In [2]:
INTERMEDIATE_PATH = Path(".").resolve() / "analysis_output"
FIELD_STATS = INTERMEDIATE_PATH / "field_stats.parquet"

# set up directory for codecarbon logs
EMISSIONS_LOG_PATH = INTERMEDIATE_PATH / "emissions_logs"
EMISSIONS_LOG_PATH.mkdir(parents=True, exist_ok=True)

MEASUREMENT_GROUPS = list(measurement_fields())  # not collapsed during induction
GROUPS = ["__date__", "__price__", *MEASUREMENT_GROUPS]


DATE_FIELDS = pl.col("field_type").is_in(date_fields())

PRICE_FIELDS = list(monetary_fields())
MEASUREMENT_FIELDS = list(measurement_fields())
STRUCTURED_FIELDS = pl.col("field_type").is_in(
    PRICE_FIELDS + MEASUREMENT_FIELDS)

PATTERN_FIELDS = DATE_FIELDS | STRUCTURED_FIELDS

TEXT_FIELDS = pl.col("field_type").is_in(free_text_fields())
TARGET_FIELDS = (
    (TEXT_FIELDS | pl.col("field_type").is_in(vocab_fields()))
    & ~PATTERN_FIELDS
)
PREFILTER = (pl.col("digit_chars") > 0) & (pl.col("char_count") >= 3)

_fs = pl.scan_parquet(FIELD_STATS).filter(pl.col("merged_pattern").is_not_null())

## 1. Per-group token lexicon

Letter-run tokens frequent in a group's structured values and rare in prose. They fill the letter slots of the probes.

In [3]:
MIN_TOKEN_COUNT = 50
MIN_LIFT = 20.0
MAX_TOKENS = 40
PROSE_SAMPLE_N = 300_000
ROMAN = {"i", "ii", "iii", "iv", "v", "vi", "vii", "viii", "ix", "x", "xi", "xii"}

group_tok = (
    _fs.select("merge_group", tok=pl.col("value").str.to_lowercase().str.extract_all(r"[a-z]+"))
    .explode("tok", empty_as_null=True).drop_nulls("tok")
    .group_by("merge_group", "tok").len("n")
    .with_columns(rate=pl.col("n") / pl.col("n").sum().over("merge_group"))
    .collect(engine="streaming")
)

prose_tok = (
    pl.scan_parquet(FIELD_STATS)
    .filter(TEXT_FIELDS & pl.col("value").is_not_null())
    .head(PROSE_SAMPLE_N)
    .select(tok=pl.col("value").str.to_lowercase().str.extract_all(r"[a-z]+"))
    .explode("tok", empty_as_null=True).drop_nulls("tok")
    .group_by("tok").len("pn")
    .collect(engine="streaming")
    .with_columns(prate=(pl.col("pn") + 1) / pl.col("pn").sum())
)

lexicon = (
    group_tok.join(prose_tok.select("tok", "prate"), on="tok", how="left")
    .with_columns(prate=pl.col("prate").fill_null(prose_tok["prate"].min()))
    .with_columns(lift=pl.col("rate") / pl.col("prate"))
    .filter((pl.col("n") >= MIN_TOKEN_COUNT) & (pl.col("lift") >= MIN_LIFT))
    .sort("n", descending=True)
    .group_by("merge_group", maintain_order=True).head(MAX_TOKENS)
    .filter(~(pl.col("merge_group").eq("__date__") & pl.col("tok").is_in(ROMAN)))
)

lexicon.group_by("merge_group").agg(pl.col("tok")).write_json(INTERMEDIATE_PATH / "lexicon_candidates.json")

LEX_ALT = {
    r["merge_group"]: "|".join(
        re.escape(t) for t in sorted(r["tok"], key=len, reverse=True))
    for r in pl.read_json(INTERMEDIATE_PATH / "lexicon.json").iter_rows(named=True)
}

## 2. Probe compilation

A shape becomes a probe only if it contains a lexicon slot or an anchor literal.

In [4]:
MIN_SLOTS, MIN_DIGIT_SLOTS = 2, 1
PROBE_COVERAGE = 0.75
ANCHORS = set("£$€×:")

_TOKEN_RE = re.compile(r"([dsS])(?:{(\d+)(?:,(\d+))?})?|(.)")


def to_probe(sig: str, lex_alt: str) -> str | None:
    tokens = []
    for m in _TOKEN_RE.finditer(sig):
        kind, lo, hi, lit = m.groups()
        if kind == "d":
            lo = int(lo) if lo else 1
            hi = int(hi) if hi else lo
            tokens.append(("c", rf"\d{{{lo}}}" if lo == hi else rf"\d{{{lo},{hi}}}"))
        elif kind:
            if not lex_alt:
                return None
            tokens.append(("x", rf"(?:{lex_alt})"))
        else:
            tokens.append(("l", re.escape(lit)))
    if not any(k == "x" for k, _ in tokens) and not any(
            k == "l" and t.lstrip("\\") in ANCHORS for k, t in tokens):
        return None
    parts = ([r"\b"] if tokens[0][0] != "l" else []) \
        + [t for _, t in tokens] \
        + ([r"\b"] if tokens[-1][0] != "l" else [])
    rx = "".join(parts)
    try:
        re.compile(rx)
    except re.error:
        return None
    return rx


skeleton = pl.col("merged_pattern").str.replace_all(r"\{[^}]*\}", "")
n_slots = skeleton.str.count_matches("[dsS]")
floors = (n_slots >= MIN_SLOTS) & (skeleton.str.count_matches("d") >= MIN_DIGIT_SLOTS)

probes = (
    _fs.group_by("merge_group", "merged_pattern").len("count")
    .filter(floors)
    .sort("count", descending=True)
    .with_columns(
        prev_cum=(pl.col("count").cum_sum().over("merge_group") - pl.col("count"))
        / pl.col("count").sum().over("merge_group"))
    .filter(pl.col("prev_cum") < PROBE_COVERAGE)
    .collect(engine="streaming")
    .with_columns(regex=pl.struct("merge_group", "merged_pattern").map_elements(
        lambda r: to_probe(r["merged_pattern"], LEX_ALT.get(r["merge_group"], "")),
        return_dtype=pl.String))
    .drop_nulls("regex")
)

MANUAL_PROBES = {
    "__price__": [r"\d{1,4}(?:[./]\d{1,2})? ?(?:guineas|gns|shillings|pence|pounds)"],
}
manual = pl.DataFrame({
    "merge_group": [g for g, rxs in MANUAL_PROBES.items() for _ in rxs],
    "merged_pattern": "manual",
    "count": 0,
    "prev_cum": 0.0,
    "regex": [rx for rxs in MANUAL_PROBES.values() for rx in rxs],
})
probes = pl.concat([probes, manual], how="vertical_relaxed")

GROUP_ALT = {
    g: "(?i)(?:" + "|".join(sorted(sub["regex"].unique(), key=len, reverse=True)) + ")"
    for (g,), sub in probes.group_by("merge_group")
}
probes.group_by("merge_group").len()

merge_group,len
str,u32
"""__date__""",9
"""spectrum/dimension""",68
"""spectrum/technical_attribute_m…",2
"""__price__""",5


## 3. Scan targets

In [5]:
with EmissionsTracker(project_name="probe_scan_targets", output_dir=str(EMISSIONS_LOG_PATH), log_level="error") as tracker:
    base = pl.scan_parquet(FIELD_STATS).filter(TARGET_FIELDS & PREFILTER)

    pl.concat([
        base.with_columns(
            candidate=pl.col("value").str.extract_all(alt),
            group=pl.lit(g))
        .filter(pl.col("candidate").list.len() > 0)
        .explode("candidate", empty_as_null=True)
        .select("node_id", "record_id", "data_source", "field_type", "group", "candidate")
        for g, alt in GROUP_ALT.items()
    ]).sink_parquet(INTERMEDIATE_PATH / "probe_candidates_raw.parquet")

[codecarbon WARNING @ 13:05:43] Multiple instances of codecarbon are allowed to run at the same time.


## 4. Parse verification

Each candidate is verified by its group's parser (`parse_dates`, `parse_dimensions`, `parse_monetary`).

In [6]:
_NUM = re.compile(r"\d+(?:[.,]\d+)?")


def _key(iso: str, end: bool) -> int:
    neg = iso.startswith("-")
    y, *rest = iso.lstrip("-").split("-")
    m = int(rest[0]) if rest else (12 if end else 1)
    d = int(rest[1]) if len(rest) > 1 else (31 if end else 1)
    return (-1 if neg else 1) * int(y) * 10_000 + m * 100 + d


def date_bounds(v: str) -> tuple[int, int] | None:
    p = parse_date(v)
    if p is None or p["date_earliest_single"] is None:
        return None
    lo = p["date_earliest_single"]
    hi = p["date_latest"] or lo
    return _key(lo, end=False), _key(hi, end=True)


def verify_measurement(v: str) -> bool:
    p = parse_dimensions(v)
    return p is not None and bool(p["measurements"])


def verify_amount(v: str) -> bool:
    """Placeholder gate: calibres and page counts are not dimensions, so no parser fits"""
    m = _NUM.search(v)
    return m is not None and float(m.group().replace(",", ".")) > 0


VERIFY = {"__price__": lambda v: parse_monetary(v) is not None,
          "spectrum/dimension": verify_measurement}
VERIFY |= {g: VERIFY.get(g, verify_amount) for g in MEASUREMENT_GROUPS}

raw = pl.scan_parquet(INTERMEDIATE_PATH / "probe_candidates_raw.parquet")
distinct = raw.select("group", "candidate").unique().collect(engine="streaming")

rows = []
for group, cand in distinct.iter_rows():
    if group == "__date__":
        b = date_bounds(cand)
        rows.append((group, cand, b is not None, *(b or (None, None))))
    else:
        rows.append((group, cand, VERIFY[group](cand), None, None))

verified = pl.DataFrame(
    rows,
    schema={
        "group": pl.String,
        "candidate": pl.String,
        "ok": pl.Boolean,
        "lo": pl.Int64,
        "hi": pl.Int64,
    },
    orient="row",
).filter(pl.col("ok")).drop("ok")
print(f"verified {len(verified):,} / {len(distinct):,} distinct candidates")

verified 33,566 / 38,959 distinct candidates


## 5. Status — dates

Each verified date is compared with the record's parsed date values: **echo**, **refine** (strictly inside a structured value, the extraction target), **additional** (disjoint) or **novel** (record has no parsed date).

In [7]:
with EmissionsTracker(project_name="probe_status_dates", output_dir=str(EMISSIONS_LOG_PATH), log_level="error") as tracker:
    rec_date_vals = (
        _fs.filter(pl.col("merge_group") == "__date__")
        .select("record_id", "value").unique()
        .collect(engine="streaming")
    )
    bounds = {
        v: b for v in rec_date_vals["value"].unique() if (b := date_bounds(v)) is not None
    }

    rec_dates = (
        rec_date_vals
        .with_columns(
            lo=pl.col("value").map_elements(lambda v: (bounds.get(v) or (None,))[0], return_dtype=pl.Int64),
            hi=pl.col("value").map_elements(lambda v: (bounds.get(v) or (None, None))[1], return_dtype=pl.Int64))
        .drop_nulls("lo")
        .select("record_id", slo="lo", shi="hi")
    )

    rec_date_fields = (
        _fs.filter(pl.col("merge_group") == "__date__")
        .select("record_id").unique()
        .with_columns(has_date_field=pl.lit(True))
    )

    date_status = (
        raw.filter(pl.col("group") == "__date__")
        .join(verified.filter(pl.col("group") == "__date__").lazy(), on=["group", "candidate"])
        .join(rec_dates.lazy(), on="record_id", how="left")
        .join(rec_date_fields, on="record_id", how="left")
        .with_columns(
            echo=((pl.col("lo") <= pl.col("slo")) & (pl.col("hi") >= pl.col("shi"))),
            refine=((pl.col("lo") >= pl.col("slo")) & (pl.col("hi") <= pl.col("shi"))
                    & ~((pl.col("lo") == pl.col("slo")) & (pl.col("hi") == pl.col("shi")))))
        .group_by("node_id", "record_id", "data_source", "field_type", "group", "candidate")
        .agg(
            n_structured=pl.col("slo").drop_nulls().len(),
            any_echo=pl.col("echo").any(),
            any_refine=pl.col("refine").any(),
            has_date_field=pl.col("has_date_field").first().fill_null(False),
            lo=pl.col("lo").first())
        .with_columns(
            status=pl.when(pl.col("n_structured") == 0)
            .then(pl.when(pl.col("has_date_field"))
                  .then(pl.lit("unparsed_field")).otherwise(pl.lit("novel")))
            .when(pl.col("any_echo")).then(pl.lit("echo"))
            .when(pl.col("any_refine")).then(pl.lit("refine"))
            .otherwise(pl.lit("additional")),
            suspect_recent=pl.col("lo") >= 20_000_101)
        .select("node_id", "record_id", "data_source", "field_type", "group",
                "candidate", "status", "suspect_recent")
    )

## 6. Status — dimensions and prices

Normalised-string comparison: **echo** if the value already sits in a structured field of the group, **refine** if the group is populated, **novel** if it is empty.

In [8]:
with EmissionsTracker(project_name="probe_status_dims_prices", output_dir=str(EMISSIONS_LOG_PATH), log_level="error") as tracker:
    norm = lambda c: c.str.to_lowercase().str.replace_all(r"\s+", "")

    rec_vals = (
        _fs.filter(pl.col("merge_group") != "__date__")
        .select("record_id", group="merge_group", vnorm=norm(pl.col("value")))
        .unique()
    )

    other_status = (
        raw.filter(pl.col("group") != "__date__")
        .join(verified.lazy().select("group", "candidate"), on=["group", "candidate"])
        .with_columns(cnorm=norm(pl.col("candidate")))
        .join(
            rec_vals.group_by("record_id", "group").agg(vnorms=pl.col("vnorm")),
            on=["record_id", "group"], how="left")
        .with_columns(
            status=pl.when(pl.col("vnorms").is_null()).then(pl.lit("novel"))
            .when(pl.col("vnorms").list.contains(pl.col("cnorm"))).then(pl.lit("echo"))
            .otherwise(pl.lit("refine")))
        .select("node_id", "record_id", "data_source", "field_type", "group", "candidate", "status")
    )

    pl.concat([date_status, other_status], how="diagonal").sink_parquet(
        INTERMEDIATE_PATH / "probe_candidates.parquet")

## 7. Sanity and review

Probes firing on more than 1% of prose cells are reported for inspection, not dropped. The stratified sample feeds threshold calibration.

In [9]:
flags = pl.scan_parquet(INTERMEDIATE_PATH / "probe_candidates.parquet")
print("candidate rows:", flags.select(pl.len()).collect(engine="streaming").item())
(
    flags.group_by("field_type", "group", "status").len()
    .sort("len", descending=True).head(30)
    .collect(engine="streaming")
)

candidate rows: 913944


field_type,group,status,len
str,str,str,u32
"""spectrum/brief_description""","""spectrum/dimension""","""refine""",127466
"""spectrum/brief_description""","""__date__""","""echo""",127347
"""spectrum/brief_description""","""__date__""","""novel""",121690
"""spectrum/object_history_note""","""__date__""","""additional""",94282
"""spectrum/brief_description""","""__date__""","""refine""",68958
…,…,…,…
"""spectrum/physical_description""","""__date__""","""novel""",3583
"""spectrum/text""","""__date__""","""unparsed_field""",3562
"""spectrum/object_history_note""","""__price__""","""novel""",3468


In [10]:
MAX_PROSE_HIT_RATE = 0.01

_prose = (
    pl.scan_parquet(FIELD_STATS)
    .filter(TEXT_FIELDS & pl.col("value").is_not_null() & (pl.col("digit_chars") > 0))
    .select("value").head(PROSE_SAMPLE_N).collect(engine="streaming")
)

hit = _prose.select(
    [pl.col("value").str.contains("(?i)" + rx).mean().alias(f"{g}::{mp}")
     for g, mp, rx in probes.select("merge_group", "merged_pattern", "regex").iter_rows()]
).row(0, named=True)

noisy = {k: v for k, v in hit.items() if v > MAX_PROSE_HIT_RATE}
print(f"{len(noisy)} noisy probes:" if noisy else "no probe exceeds the prose gate")

for name, rate in sorted(noisy.items(), key=lambda x: -x[1]):
    g, mp = name.split("::")
    rx = "(?i)" + probes.filter(
        (pl.col("merge_group") == g) & (pl.col("merged_pattern") == mp))["regex"][0]
    m = _prose.select(hit=pl.col("value").str.extract_all(rx))["hit"].explode(empty_as_null=True).drop_nulls().unique()
    ok = sum(1 for v in m if (date_bounds(v) is not None if g == "__date__" else VERIFY[g](v)))
    print(f"  {name}  prose_rate={rate:.3f}  parse_rate={ok / len(m):.2f}")

1 noisy probes:
  __date__::s{3,10} d{4}  prose_rate=0.043  parse_rate=0.96


In [11]:
REVIEW_N = 8

review = (
    flags.filter(pl.col("status") != "echo")
    .group_by("group", "status")
    .agg(sample=pl.struct("field_type", "candidate", "record_id").shuffle(seed=0).head(REVIEW_N))
    .collect(engine="streaming")
)
review.write_parquet(INTERMEDIATE_PATH / "probe_review_sample.parquet")
review

group,status,sample
str,str,list[struct[3]]
"""__date__""","""novel""","[{""spectrum/brief_description"",""September 1798"",""66ccb410-dcb5-3238-8aba-5eea5a335d5c""}, {""spectrum/brief_description"",""May 1973"",""4c2187ac-0ad8-35ed-9334-22e1791769a3""}, … {""spectrum/brief_description"",""January 2003"",""d4eee93f-5791-3313-8185-932c6e551fdd""}]"
"""spectrum/technical_attribute_m…","""novel""","[{""spectrum/brief_description"",""18 pages"",""2839cee5-9325-3cbf-8a0b-5a1d9252d172""}, {""spectrum/brief_description"",""64 pages"",""c8cf851d-55c5-3bff-91f6-8d3f4f3c4e9a""}, … {""spectrum/brief_description"",""24 pages"",""2f742379-d39d-3786-b2fc-34b9e97588a4""}]"
"""spectrum/technical_attribute_m…","""refine""","[{""spectrum/brief_description"",""50 calibre"",""d5de8c43-3fa0-326c-9717-4ef874296c1c""}, {""spectrum/brief_description"",""470 pages"",""59223a92-52f7-372e-9e4b-00733d905c36""}, … {""spectrum/brief_description"",""22 cartridge"",""cd30fce2-6f03-39fd-82e7-1266ddaf885a""}]"
"""spectrum/dimension""","""novel""","[{""spectrum/brief_description"",""x 160 mm"",""469581a0-9271-3bbe-8b90-a98f0685d496""}, {""spectrum/brief_description"",""35mm"",""b82f534e-b29f-34a1-9255-2d7b490dc811""}, … {""spectrum/brief_description"",""76 x 51cm"",""b8657255-cc12-3c50-814d-b1d23a1a9f39""}]"
"""__date__""","""additional""","[{""spectrum/object_history_note"",""July 1956"",""17fce63d-163d-32fa-8b68-ca1927168ffc""}, {""spectrum/text"",""ca. 1680"",""364d48e8-878f-3cdb-80be-501a7b404df7""}, … {""spectrum/object_history_note"",""April 2016"",""b20884bc-8e24-3819-8e9b-a4cd40bec494""}]"
"""__date__""","""refine""","[{""spectrum/text"",""April 1925"",""af0c90b4-ada7-3880-b090-55f4cd371fc4""}, {""spectrum/object_history_note"",""July 1859"",""27fb1ef2-c7c4-3a90-94a5-60ca4dd89257""}, … {""spectrum/text"",""March 1912"",""bfaa447b-ce78-3701-8add-acd2efbd1a4b""}]"
"""__price__""","""novel""","[{""spectrum/object_history_note"",""15 pounds"",""43f0aa43-2167-3feb-85cf-164d2c6dfe16""}, {""spectrum/brief_description"",""£5.13"",""76327a8a-12e5-3374-8a66-39df7abe3344""}, … {""spectrum/object_production_note"",""£15.15"",""7776c853-15e9-3689-9cf0-5ba9410597c1""}]"
"""__date__""","""unparsed_field""","[{""spectrum/object_history_note"",""July 2012"",""ef74d6e8-30db-3426-8bdc-669cbf011a32""}, {""spectrum/brief_description"",""ca. 1857"",""f4cb00ac-0949-3d8e-8b81-85ff5e5d286a""}, … {""spectrum/text"",""about 1824"",""c42425db-18ed-361f-b138-33698891b957""}]"
"""spectrum/dimension""","""refine""","[{""spectrum/brief_description"",""02cm"",""566cdbd1-d325-3082-85c2-ad532132366d""}, {""spectrum/brief_description"",""54cm"",""a6ba9460-2067-34b0-918d-e92a5d319f52""}, … {""spectrum/brief_description"",""48.3 x 39.4 cm"",""5d45a5a4-211f-3fe0-a9f8-3b44abb5a561""}]"
